# Entropy Analysis and Visualization

Explore local entropy patterns in FPGA bitstreams to identify potential anomalies.

## Goals

1. Load preprocessed bitstream images
2. Compute local Shannon entropy
3. Visualize entropy maps
4. Compare clean vs Trojan-infected designs

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.io import imread
from scipy.stats import entropy

from preprocess.visualize_entropy import compute_local_entropy

## Load sample images

In [ ]:
# TODO: Replace with actual image paths
clean_img_path = Path("../dataset/images/clean_001.png")
trojan_img_path = Path("../dataset/images/trojan_001.png")

if clean_img_path.exists():
    clean_img = imread(clean_img_path, as_gray=True)
    clean_img = (clean_img * 255).astype(np.uint8)
else:
    print("Creating synthetic clean image")
    clean_img = np.random.randint(0, 2, size=(1000, 1024), dtype=np.uint8) * 255

if trojan_img_path.exists():
    trojan_img = imread(trojan_img_path, as_gray=True)
    trojan_img = (trojan_img * 255).astype(np.uint8)
else:
    print("Creating synthetic Trojan image")
    trojan_img = clean_img.copy()
    # Insert high-entropy patch (simulated Trojan)
    trojan_img[400:450, 500:550] = np.random.randint(0, 256, size=(50, 50), dtype=np.uint8)

## Compute entropy maps

In [ ]:
window_size = 16

clean_entropy = compute_local_entropy(clean_img, window_size)
trojan_entropy = compute_local_entropy(trojan_img, window_size)

print(f"Clean entropy map shape: {clean_entropy.shape}")
print(f"Trojan entropy map shape: {trojan_entropy.shape}")

## Visualize comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Clean design
axes[0, 0].imshow(clean_img, cmap='gray')
axes[0, 0].set_title('Clean Bitstream')
axes[0, 0].axis('off')

axes[0, 1].imshow(clean_entropy, cmap='hot')
axes[0, 1].set_title('Clean Entropy Map')
axes[0, 1].axis('off')

axes[0, 2].hist(clean_entropy.flatten(), bins=50)
axes[0, 2].set_title('Clean Entropy Distribution')
axes[0, 2].set_xlabel('Entropy')
axes[0, 2].set_ylabel('Count')

# Trojan design
axes[1, 0].imshow(trojan_img, cmap='gray')
axes[1, 0].set_title('Trojan Bitstream')
axes[1, 0].axis('off')

axes[1, 1].imshow(trojan_entropy, cmap='hot')
axes[1, 1].set_title('Trojan Entropy Map')
axes[1, 1].axis('off')

axes[1, 2].hist(trojan_entropy.flatten(), bins=50)
axes[1, 2].set_title('Trojan Entropy Distribution')
axes[1, 2].set_xlabel('Entropy')
axes[1, 2].set_ylabel('Count')

plt.tight_layout()
plt.show()

## Statistical comparison

In [ ]:
print("Clean entropy statistics:")
print(f"  Mean: {clean_entropy.mean():.4f}")
print(f"  Std:  {clean_entropy.std():.4f}")
print(f"  Max:  {clean_entropy.max():.4f}")
print()
print("Trojan entropy statistics:")
print(f"  Mean: {trojan_entropy.mean():.4f}")
print(f"  Std:  {trojan_entropy.std():.4f}")
print(f"  Max:  {trojan_entropy.max():.4f}")